In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
DATA_PATH = Path(
    r"C:\Users\kalav\OneDrive\Desktop\EMIPredict-AI\data\processed\emi_prediction_dataset_cleaned.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Shape: (404800, 27)


,age,gender,marital_status,education,monthly_salary,employment_type,years_of_employment,company_type,house_type,monthly_rent,...,existing_loans,current_emi_amount,credit_score,bank_balance,emergency_fund,emi_scenario,requested_amount,requested_tenure,emi_eligibility,max_monthly_emi
0,38.0,Female,Married,Professional,82600.0,Private,0.9,Mid-size,Rented,20000.0,...,Yes,23700.0,660.0,303200.0,70200.0,Personal Loan EMI,850000.0,15,Not_Eligible,500.0
1,38.0,Female,Married,Graduate,21500.0,Private,7.0,MNC,Family,0.0,...,Yes,4100.0,714.0,92500.0,26900.0,E-commerce Shopping EMI,128000.0,19,Not_Eligible,700.0
2,38.0,Male,Married,Professional,86100.0,Private,5.8,Startup,Own,0.0,...,No,0.0,650.0,672100.0,324200.0,Education EMI,306000.0,16,Eligible,27775.0
3,58.0,Female,Married,High School,66800.0,Private,2.2,Mid-size,Own,0.0,...,No,0.0,685.0,440900.0,178100.0,Vehicle EMI,304000.0,83,Eligible,16170.0
4,48.0,Female,Married,Professional,57300.0,Private,3.4,Mid-size,Family,0.0,...,No,0.0,770.0,97300.0,28200.0,Home Appliances EMI,252000.0,7,Not_Eligible,500.0


In [4]:
print("\nMissing values:")
print(df.isnull().sum().sum())


Missing values:
0


In [5]:
print("\nDuplicate rows:")
print(df.duplicated().sum())


Duplicate rows:
0


In [6]:
print("\nData types:")
print(df.dtypes)


Data types:
age                       float64
gender                     object
marital_status             object
education                  object
monthly_salary            float64
employment_type            object
years_of_employment       float64
company_type               object
house_type                 object
monthly_rent              float64
family_size                 int64
dependents                  int64
school_fees               float64
college_fees              float64
travel_expenses           float64
groceries_utilities       float64
other_monthly_expenses    float64
existing_loans             object
current_emi_amount        float64
credit_score              float64
bank_balance              float64
emergency_fund            float64
emi_scenario               object
requested_amount          float64
requested_tenure            int64
emi_eligibility            object
max_monthly_emi           float64
dtype: object


In [7]:
CLASSIFICATION_TARGET = "emi_eligibility"

REGRESSION_TARGET = "max_monthly_emi"

print("Classification Target:", CLASSIFICATION_TARGET)
print("Regression Target:", REGRESSION_TARGET)

Classification Target: emi_eligibility
Regression Target: max_monthly_emi


In [8]:
print("CLASSIFICATION TARGET")


display(df[CLASSIFICATION_TARGET].value_counts())

print("\nREGRESSION TARGET")


display(df[REGRESSION_TARGET].describe().round(2))

CLASSIFICATION TARGET


emi_eligibility
Not_Eligible    312868
Eligible         74444
High_Risk        17488
Name: count, dtype: int64


REGRESSION TARGET


count    404800.00
mean       6763.60
std        7741.26
min         500.00
25%         500.00
50%        4211.20
75%        9792.00
max       91040.40
Name: max_monthly_emi, dtype: float64

In [9]:
categorical_features = [
    "gender",
    "marital_status",
    "education",
    "employment_type",
    "company_type",
    "house_type",
    "existing_loans",
    "emi_scenario"
]

numerical_features = [
    "age",
    "monthly_salary",
    "years_of_employment",
    "monthly_rent",
    "family_size",
    "dependents",
    "school_fees",
    "college_fees",
    "travel_expenses",
    "groceries_utilities",
    "other_monthly_expenses",
    "current_emi_amount",
    "credit_score",
    "bank_balance",
    "emergency_fund",
    "requested_amount",
    "requested_tenure"
]

print("Categorical features:", len(categorical_features))
print("Numerical features:", len(numerical_features))

print("\nCategorical:")
print(categorical_features)

print("\nNumerical:")
print(numerical_features)

Categorical features: 8
Numerical features: 17

Categorical:
['gender', 'marital_status', 'education', 'employment_type', 'company_type', 'house_type', 'existing_loans', 'emi_scenario']

Numerical:
['age', 'monthly_salary', 'years_of_employment', 'monthly_rent', 'family_size', 'dependents', 'school_fees', 'college_fees', 'travel_expenses', 'groceries_utilities', 'other_monthly_expenses', 'current_emi_amount', 'credit_score', 'bank_balance', 'emergency_fund', 'requested_amount', 'requested_tenure']


In [10]:
excluded_columns = [
    CLASSIFICATION_TARGET,
    REGRESSION_TARGET
]

print("Columns excluded from model features:")
for col in excluded_columns:
    print(" -", col)

Columns excluded from model features:
 - emi_eligibility
 - max_monthly_emi


In [11]:
expense_columns = [
    "monthly_rent",
    "school_fees",
    "college_fees",
    "travel_expenses",
    "groceries_utilities",
    "other_monthly_expenses"
]

df["total_living_expenses"] = df[expense_columns].sum(axis=1)

df["total_monthly_commitments"] = (
    df["total_living_expenses"]
    + df["current_emi_amount"]
)

df["disposable_income"] = (
    df["monthly_salary"]
    - df["total_monthly_commitments"]
)

print("Financial features created successfully!")

Financial features created successfully!


In [12]:
# Avoid division by zero
salary_safe = df["monthly_salary"].replace(0, np.nan)

df["expense_to_income_ratio"] = (
    df["total_living_expenses"] / salary_safe
)

df["commitment_to_income_ratio"] = (
    df["total_monthly_commitments"] / salary_safe
)

df["current_emi_to_income_ratio"] = (
    df["current_emi_amount"] / salary_safe
)

df["requested_amount_to_income"] = (
    df["requested_amount"] / salary_safe
)

df["emergency_fund_to_income"] = (
    df["emergency_fund"] / salary_safe
)

print("Financial ratios created successfully!")

Financial ratios created successfully!


In [13]:
engineered_features = [
    "total_living_expenses",
    "total_monthly_commitments",
    "disposable_income",
    "expense_to_income_ratio",
    "commitment_to_income_ratio",
    "current_emi_to_income_ratio",
    "requested_amount_to_income",
    "emergency_fund_to_income"
]

display(
    df[engineered_features]
    .describe()
    .T
    .round(3)
)

,count,mean,std,min,25%,50%,75%,max
total_living_expenses,404800.0,40125.148,20307.582,3400.000,25400.000,36700.000,51100.000,207500.000
total_monthly_commitments,404800.0,44668.556,23072.508,3600.000,28000.000,40600.000,56800.000,247800.000
disposable_income,404800.0,14833.380,37607.524,-161883.000,400.000,10600.000,22400.000,485540.000
expense_to_income_ratio,404800.0,0.784,0.579,0.011,0.531,0.701,0.889,26.777
commitment_to_income_ratio,404800.0,0.868,0.637,0.011,0.608,0.780,0.991,31.292
current_emi_to_income_ratio,404800.0,0.084,0.142,0.000,0.000,0.000,0.175,7.704
requested_amount_to_income,404800.0,8.717,11.546,0.023,2.188,4.787,10.837,294.557
emergency_fund_to_income,404800.0,1.779,1.722,0.005,0.854,1.543,2.395,90.130


In [14]:
print("Missing values:")
display(df[engineered_features].isnull().sum())

print("\nInfinite values:")
display(
    np.isinf(df[engineered_features])
    .sum()
)

Missing values:


total_living_expenses          0
total_monthly_commitments      0
disposable_income              0
expense_to_income_ratio        0
commitment_to_income_ratio     0
current_emi_to_income_ratio    0
requested_amount_to_income     0
emergency_fund_to_income       0
dtype: int64


Infinite values:


total_living_expenses          0
total_monthly_commitments      0
disposable_income              0
expense_to_income_ratio        0
commitment_to_income_ratio     0
current_emi_to_income_ratio    0
requested_amount_to_income     0
emergency_fund_to_income       0
dtype: int64

In [15]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Infinite values replaced.")

Infinite values replaced.


In [16]:
y_classification = df["emi_eligibility"]

print("Classification Target:")
print(y_classification.value_counts())

print("\nPercentages:")
print(
    y_classification.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Classification Target:
emi_eligibility
Not_Eligible    312868
Eligible         74444
High_Risk        17488
Name: count, dtype: int64

Percentages:
emi_eligibility
Not_Eligible    77.29
Eligible        18.39
High_Risk        4.32
Name: proportion, dtype: float64


In [17]:
classification_exclude = [
    "emi_eligibility",
    "max_monthly_emi"
]

classification_features = [
    col for col in df.columns
    if col not in classification_exclude
]

X_classification = df[classification_features].copy()
y_classification = df["emi_eligibility"].copy()

print("Classification feature shape:", X_classification.shape)
print("Classification target shape:", y_classification.shape)

Classification feature shape: (404800, 33)
Classification target shape: (404800,)


In [18]:
regression_exclude = [
    "emi_eligibility",
    "max_monthly_emi"
]

regression_features = [
    col for col in df.columns
    if col not in regression_exclude
]

X_regression = df[regression_features].copy()
y_regression = df["max_monthly_emi"].copy()

print("Regression feature shape:", X_regression.shape)
print("Regression target shape:", y_regression.shape)

Regression feature shape: (404800, 33)
Regression target shape: (404800,)


In [19]:
print("Categorical features:")
print(X_classification.select_dtypes(include=["object"]).columns.tolist())

print("\nNumerical features:")
print(X_classification.select_dtypes(include=["number"]).columns.tolist())

Categorical features:
['gender', 'marital_status', 'education', 'employment_type', 'company_type', 'house_type', 'existing_loans', 'emi_scenario']

Numerical features:
['age', 'monthly_salary', 'years_of_employment', 'monthly_rent', 'family_size', 'dependents', 'school_fees', 'college_fees', 'travel_expenses', 'groceries_utilities', 'other_monthly_expenses', 'current_emi_amount', 'credit_score', 'bank_balance', 'emergency_fund', 'requested_amount', 'requested_tenure', 'total_living_expenses', 'total_monthly_commitments', 'disposable_income', 'expense_to_income_ratio', 'commitment_to_income_ratio', 'current_emi_to_income_ratio', 'requested_amount_to_income', 'emergency_fund_to_income']


In [20]:
numeric_df = X_classification.select_dtypes(
    include=["number"]
)

corr_matrix = numeric_df.corr().abs()

upper_triangle = corr_matrix.where(
    np.triu(
        np.ones(corr_matrix.shape),
        k=1
    ).astype(bool)
)

high_correlations = (
    upper_triangle
    .stack()
    .reset_index()
)

high_correlations.columns = [
    "Feature 1",
    "Feature 2",
    "Correlation"
]

high_correlations = high_correlations[
    high_correlations["Correlation"] >= 0.90
].sort_values(
    "Correlation",
    ascending=False
)

display(high_correlations)

,Feature 1,Feature 2,Correlation
90,family_size,dependents,1.000000
290,expense_to_income_ratio,commitment_to_income_ratio,0.977239
272,total_living_expenses,total_monthly_commitments,0.955346


In [21]:
FEATURED_PATH = Path(
    r"C:\Users\kalav\OneDrive\Desktop\EMIPredict-AI\data\processed\emi_prediction_dataset_featured.csv"
)

df.to_csv(FEATURED_PATH, index=False)

print("Feature-engineered dataset saved successfully!")
print("Location:", FEATURED_PATH)
print("Shape:", df.shape)

Feature-engineered dataset saved successfully!
Location: C:\Users\kalav\OneDrive\Desktop\EMIPredict-AI\data\processed\emi_prediction_dataset_featured.csv
Shape: (404800, 35)


In [22]:
CLASSIFICATION_TARGET = "emi_eligibility"

y_classification = df[CLASSIFICATION_TARGET].copy()

print("Classification Target:", CLASSIFICATION_TARGET)

print("\nTarget Distribution:")
display(y_classification.value_counts())

print("\nTarget Percentage:")
display(
    y_classification
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Classification Target: emi_eligibility

Target Distribution:


emi_eligibility
Not_Eligible    312868
Eligible         74444
High_Risk        17488
Name: count, dtype: int64


Target Percentage:


emi_eligibility
Not_Eligible    77.29
Eligible        18.39
High_Risk        4.32
Name: proportion, dtype: float64

In [23]:
classification_exclude = [
    "emi_eligibility",   # Classification target
    "max_monthly_emi"    # Regression target / possible leakage
]

print("Excluded columns:")
for col in classification_exclude:
    print("-", col)

Excluded columns:
- emi_eligibility
- max_monthly_emi


In [24]:
classification_features = [
    col
    for col in df.columns
    if col not in classification_exclude
]

X_classification = df[classification_features].copy()

print("Classification Dataset")
print("=" * 60)

print("X shape:", X_classification.shape)
print("y shape:", y_classification.shape)

print("\nNumber of features:", len(classification_features))

Classification Dataset
X shape: (404800, 33)
y shape: (404800,)

Number of features: 33


In [25]:
print("Classification Features:")

for i, col in enumerate(classification_features, start=1):
    print(f"{i}. {col}")

Classification Features:
1. age
2. gender
3. marital_status
4. education
5. monthly_salary
6. employment_type
7. years_of_employment
8. company_type
9. house_type
10. monthly_rent
11. family_size
12. dependents
13. school_fees
14. college_fees
15. travel_expenses
16. groceries_utilities
17. other_monthly_expenses
18. existing_loans
19. current_emi_amount
20. credit_score
21. bank_balance
22. emergency_fund
23. emi_scenario
24. requested_amount
25. requested_tenure
26. total_living_expenses
27. total_monthly_commitments
28. disposable_income
29. expense_to_income_ratio
30. commitment_to_income_ratio
31. current_emi_to_income_ratio
32. requested_amount_to_income
33. emergency_fund_to_income


In [26]:
classification_numeric_features = (
    X_classification
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

classification_categorical_features = (
    X_classification
    .select_dtypes(include=["object"])
    .columns
    .tolist()
)

print("Numerical Features:", len(classification_numeric_features))
print(classification_numeric_features)

print("\nCategorical Features:", len(classification_categorical_features))
print(classification_categorical_features)

Numerical Features: 25
['age', 'monthly_salary', 'years_of_employment', 'monthly_rent', 'family_size', 'dependents', 'school_fees', 'college_fees', 'travel_expenses', 'groceries_utilities', 'other_monthly_expenses', 'current_emi_amount', 'credit_score', 'bank_balance', 'emergency_fund', 'requested_amount', 'requested_tenure', 'total_living_expenses', 'total_monthly_commitments', 'disposable_income', 'expense_to_income_ratio', 'commitment_to_income_ratio', 'current_emi_to_income_ratio', 'requested_amount_to_income', 'emergency_fund_to_income']

Categorical Features: 8
['gender', 'marital_status', 'education', 'employment_type', 'company_type', 'house_type', 'existing_loans', 'emi_scenario']


In [28]:
print("CLASSIFICATION LEAKAGE CHECK")


print("\nTarget present in X?")
print(CLASSIFICATION_TARGET in X_classification.columns)

print("\nRegression target present in X?")
print("max_monthly_emi" in X_classification.columns)

print("\nX shape:", X_classification.shape)
print("y shape:", y_classification.shape)

CLASSIFICATION LEAKAGE CHECK

Target present in X?
False

Regression target present in X?
False

X shape: (404800, 33)
y shape: (404800,)


In [29]:
print("Classification Features:")
display(X_classification.head())

print("\nClassification Target:")
display(y_classification.head())

Classification Features:


,age,gender,marital_status,education,monthly_salary,employment_type,years_of_employment,company_type,house_type,monthly_rent,...,requested_amount,requested_tenure,total_living_expenses,total_monthly_commitments,disposable_income,expense_to_income_ratio,commitment_to_income_ratio,current_emi_to_income_ratio,requested_amount_to_income,emergency_fund_to_income
0,38.0,Female,Married,Professional,82600.0,Private,0.9,Mid-size,Rented,20000.0,...,850000.0,15,59900.0,83600.0,-1000.0,0.725182,1.012107,0.286925,10.290557,0.849879
1,38.0,Female,Married,Graduate,21500.0,Private,7.0,MNC,Family,0.0,...,128000.0,19,15400.0,19500.0,2000.0,0.716279,0.906977,0.190698,5.953488,1.251163
2,38.0,Male,Married,Professional,86100.0,Private,5.8,Startup,Own,0.0,...,306000.0,16,35600.0,35600.0,50500.0,0.413473,0.413473,0.000000,3.554007,3.765389
3,58.0,Female,Married,High School,66800.0,Private,2.2,Mid-size,Own,0.0,...,304000.0,83,37400.0,37400.0,29400.0,0.559880,0.559880,0.000000,4.550898,2.666168
4,48.0,Female,Married,Professional,57300.0,Private,3.4,Mid-size,Family,0.0,...,252000.0,7,58600.0,58600.0,-1300.0,1.022688,1.022688,0.000000,4.397906,0.492147



Classification Target:


0    Not_Eligible
1    Not_Eligible
2        Eligible
3        Eligible
4    Not_Eligible
Name: emi_eligibility, dtype: object

In [30]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X_classification,
    y_classification,
    test_size=0.20,
    random_state=42,
    stratify=y_classification
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

print("\nTraining target distribution:")
display(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTesting target distribution:")
display(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Training data shape: (323840, 33)
Testing data shape: (80960, 33)

Training target distribution:


emi_eligibility
Not_Eligible    77.29
Eligible        18.39
High_Risk        4.32
Name: proportion, dtype: float64


Testing target distribution:


emi_eligibility
Not_Eligible    77.29
Eligible        18.39
High_Risk        4.32
Name: proportion, dtype: float64

In [32]:
numeric_features = X_classification.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_classification.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

Numerical features: 25
['age', 'monthly_salary', 'years_of_employment', 'monthly_rent', 'family_size', 'dependents', 'school_fees', 'college_fees', 'travel_expenses', 'groceries_utilities', 'other_monthly_expenses', 'current_emi_amount', 'credit_score', 'bank_balance', 'emergency_fund', 'requested_amount', 'requested_tenure', 'total_living_expenses', 'total_monthly_commitments', 'disposable_income', 'expense_to_income_ratio', 'commitment_to_income_ratio', 'current_emi_to_income_ratio', 'requested_amount_to_income', 'emergency_fund_to_income']

Categorical features: 8
['gender', 'marital_status', 'education', 'employment_type', 'company_type', 'house_type', 'existing_loans', 'emi_scenario']


In [33]:

numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

print("Numerical pipeline created.")
print("Categorical pipeline created.")

Numerical pipeline created.
Categorical pipeline created.


In [34]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ],
    remainder="drop"
)

print("Preprocessor created successfully!")

Preprocessor created successfully!


In [35]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Training data after preprocessing:", X_train_processed.shape)
print("Testing data after preprocessing:", X_test_processed.shape)

Training data after preprocessing: (323840, 51)
Testing data after preprocessing: (80960, 51)


In [36]:
print("Processed training data type:")
print(type(X_train_processed))

print("\nProcessed testing data type:")
print(type(X_test_processed))

print("\nTraining shape:")
print(X_train_processed.shape)

print("\nTesting shape:")
print(X_test_processed.shape)

print("\nAny NaN in training data?")
print(np.isnan(X_train_processed).sum())

print("\nAny NaN in testing data?")
print(np.isnan(X_test_processed).sum())

Processed training data type:
<class 'numpy.ndarray'>

Processed testing data type:
<class 'numpy.ndarray'>

Training shape:
(323840, 51)

Testing shape:
(80960, 51)

Any NaN in training data?
0

Any NaN in testing data?
0


In [37]:
import joblib

ARTIFACTS_PATH = Path(
    r"C:\Users\kalav\OneDrive\Desktop\EMIPredict-AI\artifacts"
)

ARTIFACTS_PATH.mkdir(
    parents=True,
    exist_ok=True
)

preprocessor_path = ARTIFACTS_PATH / "classification_preprocessor.pkl"

joblib.dump(
    preprocessor,
    preprocessor_path
)

print("Classification preprocessor saved!")
print("Location:", preprocessor_path)

Classification preprocessor saved!
Location: C:\Users\kalav\OneDrive\Desktop\EMIPredict-AI\artifacts\classification_preprocessor.pkl


In [38]:
print("CLASSIFICATION DATA READY FOR MODEL TRAINING")


print("X_train_processed:", X_train_processed.shape)
print("X_test_processed :", X_test_processed.shape)
print("y_train           :", y_train.shape)
print("y_test            :", y_test.shape)

CLASSIFICATION DATA READY FOR MODEL TRAINING
X_train_processed: (323840, 51)
X_test_processed : (80960, 51)
y_train           : (323840,)
y_test            : (80960,)
